# Parameter tester

Quick, no-subsampling playground: **load** a dataset, optionally **add** signature targets / cancer factors, then **fit once** and look at the betas. Nothing is cached.

## 1. Load

In [1]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.least_squares import fit_gene_betas, rank_coefficients

In [2]:
# Pick a dataset (any key in CONFIGS: 'cervical' | 'breast' | 'colon').
cfg = CONFIGS['breast']
adata = sc.read_h5ad(cfg.adata_path)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')

# Cell selection + fit settings default to the dataset's config; override inline to test.
ANNOT_COL = cfg.annot_col or cfg.cell_type_col
CELL_TYPE = cfg.cell_type or (cfg.t_cell_types[0] if cfg.t_cell_types else None)
METHOD, PENALTY, L1_RATIO, STANDARDIZE = cfg.method, cfg.penalty, cfg.l1_ratio, cfg.standardize
print('annot_col =', ANNOT_COL, '| cell_type =', CELL_TYPE,
      '| method =', METHOD, '| penalty =', PENALTY)

170045 cells | 18028 built genes
annot_col = cell_type | cell_type = T Lymphocytes | method = l1 | penalty = 0.005


## 2. (optional) add signature targets / cancer factors

Uncomment what you want. Leave both commented to fit a plain gene.

In [8]:
library = None  # lazy: fetched once below if any optional block needs it

# --- OPTION A: add cancer gene-set signatures as spatial-neighborhood FACTORS (cancers) ---
# library = library or sh.load_msigdb(cfg.dbver, cfg.categories)
# cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
# cancer_sets = {k: v for k, v in cancer_sets.items() if v}
# sh.add_cancer_factors(adata, cancer_sets, annot_col=cfg.cell_type_col, add_proximity=True)

# --- OPTION B: build a signature pseudo-gene TARGET (instead of a raw gene) ---
library = library or sh.load_msigdb(cfg.dbver, cfg.categories)
# SIG = 'HALLMARK_IL2_STAT5_SIGNALING'
SIG = 'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY'
genes = [g for g in library.get(SIG, []) if g in var]
adata = sh.add_gene_signature(adata, SIG, positive=genes, keep_when_overlap='factor', verbose=True)
TARGET = SIG

# Default TARGET: a raw built gene. Edit to any gene in adata.uns['x_genes'].
# TARGET = cfg.single_gene or adata.uns['x_genes'][0]
print('TARGET =', TARGET)
TARGET = 'FOXP3'


KeyboardInterrupt



In [9]:
TARGET = 'FOXP3'

## 3. Fit once (no subsampling) and view the betas

In [10]:
betas_ridge = fit_gene_betas(adata, genes=[TARGET], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                       metabolites=[], custom='all', method='ridge', penalty=1000,
                       l1_ratio=L1_RATIO, standardize=STANDARDIZE)
rank_coefficients(betas_ridge).head(60)

,gene,factor,group,beta,r,r2,model_r2,n_cells
1655,FOXP3,CD80$CTLA4,lr,0.022610,0.215725,4.653733e-02,0.33855,8018
1763,FOXP3,ICOSLG$CTLA4,lr,0.019133,0.166443,2.770329e-02,0.33855,8018
549,FOXP3,CCL1$CCR8,lr,0.013239,0.142599,2.033461e-02,0.33855,8018
1434,FOXP3,LAMB2$ITGAV,lr,0.010185,0.030775,9.470800e-04,0.33855,8018
783,FOXP3,CD70$CD27,lr,0.010148,0.061300,3.757729e-03,0.33855,8018
39,FOXP3,HIVEP3,tf,0.009909,0.051583,2.660809e-03,0.33855,8018
1252,FOXP3,THBS2$ITGB1,lr,0.009744,0.027045,7.314380e-04,0.33855,8018
1443,FOXP3,COL4A2$ITGAV,lr,-0.009338,-0.005959,3.551038e-05,0.33855,8018
1503,FOXP3,COL4A5$SDC1,lr,0.009265,0.067294,4.528496e-03,0.33855,8018
606,FOXP3,CXCL12$CXCR4,lr,-0.008561,-0.064138,4.113704e-03,0.33855,8018


In [11]:
br = betas_ridge['beta']
print(f'{np.sum(np.abs(br)), np.sum(np.power(br, 2))}')

(5.03617, 0.018044341)


In [12]:
betas_l1 = fit_gene_betas(adata, genes=[TARGET], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                       metabolites=[], custom='all', method='l1', penalty=0.005,
                       l1_ratio=L1_RATIO, standardize=STANDARDIZE)
rank_coefficients(betas_l1).head(60)

,gene,factor,group,beta,r,r2,model_r2,n_cells
1655,FOXP3,CD80$CTLA4,lr,0.024709,0.215725,0.046537,0.115293,8018
1763,FOXP3,ICOSLG$CTLA4,lr,0.016321,0.166443,0.027703,0.115293,8018
549,FOXP3,CCL1$CCR8,lr,0.014393,0.142599,0.020335,0.115293,8018
1503,FOXP3,COL4A5$SDC1,lr,0.009241,0.067294,0.004528,0.115293,8018
1657,FOXP3,CD86$CTLA4,lr,0.008865,0.170074,0.028925,0.115293,8018
768,FOXP3,LTA$TNFRSF1B,lr,0.008071,0.123729,0.015309,0.115293,8018
1827,FOXP3,HLA-DQA2$CD4,lr,0.006173,0.064756,0.004193,0.115293,8018
606,FOXP3,CXCL12$CXCR4,lr,-0.006111,-0.064138,0.004114,0.115293,8018
1671,FOXP3,CD209$CEACAM1,lr,0.004842,0.054225,0.002940,0.115293,8018
34,FOXP3,GATA3,tf,0.004667,0.063294,0.004006,0.115293,8018


In [13]:
bl = betas_l1['beta']
print(f'{np.sum(np.abs(bl)), np.sum(np.power(bl, 2))}')

(0.22379251, 0.0016845639)
